**Purpose:** Aggregate v1 cross-validation results (BERT / TF-IDF / sector embeddings).

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [6]:
import json
import pandas as pd
import os

In [ ]:
bertemb = [x for x in os.listdir("cv_results_v1") if x.startswith("bertemb") and x.endswith(".json")]
sectoremb = [x for x in os.listdir("cv_results_v1") if x.startswith("sectoremb") and x.endswith(".json")]
tfidfemb = [x for x in os.listdir("cv_results_v1") if x.startswith("tfidfemb") and x.endswith(".json")]

len(bertemb), len(sectoremb), len(tfidfemb)

(3, 2, 4)

In [ ]:
bertemb_results = {}
for file in bertemb:
    with open(os.path.join("cv_results_v1", file), "r") as f:
        data = json.load(f)
        for hyperparam in data:
            bertemb_results[hyperparam] = data[hyperparam]

sectoremb_results = {}
for file in sectoremb:
    with open(os.path.join("cv_results_v1", file), "r") as f:
        data = json.load(f)
        for hyperparam in data:
            sectoremb_results[hyperparam] = data[hyperparam]

tfidfemb_results = {}
for file in tfidfemb:
    with open(os.path.join("cv_results_v1", file), "r") as f:
        data = json.load(f)
        for hyperparam in data:
            tfidfemb_results[hyperparam] = data[hyperparam]

len(bertemb_results), len(sectoremb_results), len(tfidfemb_results)

(22, 13, 25)

In [9]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

def plot_global_macro_f1(data, title="GLOBAL Macro-F1 across folds (Train vs Validation)"):
    records = []

    for cfg, cfg_data in data.items():
        cols = cfg_data["columns"]
        data = cfg_data["data"]

        # find GLOBAL row
        global_row = next(row for row in data if row[0] == "GLOBAL")

        for fold in range(2, 7):
            rec = {
                "config": cfg,
                "fold": fold,
                "train": global_row[cols.index(f"MacroF1_train_{fold}")],
                "val":   global_row[cols.index(f"MacroF1_val_{fold}")]
            }
            records.append(rec)

    df = pd.DataFrame(records)



    colors = px.colors.qualitative.Set2  # nice, readable palette

    fig = go.Figure()

    for i, cfg in enumerate(df["config"].unique()):
        d = df[df["config"] == cfg]
        color = colors[i % len(colors)]

        # Validation (solid)
        fig.add_trace(go.Scatter(
            x=d["fold"],
            y=d["val"],
            mode="lines+markers",
            name=f"{cfg} | val",
            legendgroup=cfg,
            line=dict(color=color)
        ))

        # Train (dashed, same color)
        fig.add_trace(go.Scatter(
            x=d["fold"],
            y=d["train"],
            mode="lines+markers",
            name=f"{cfg} | train",
            legendgroup=cfg,
            line=dict(color=color, dash="dash"),
            showlegend=False
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Fold",
        yaxis_title="Macro-F1",
        hovermode="x unified"
    )

    fig.show()

In [10]:
plot_global_macro_f1(bertemb_results, title="BERT Embeddings: GLOBAL Macro-F1 across folds (Train vs Validation)")
plot_global_macro_f1(sectoremb_results, title="Sector Embeddings: GLOBAL Macro-F1 across folds (Train vs Validation)")
plot_global_macro_f1(tfidfemb_results, title="TF-IDF Embeddings: GLOBAL Macro-F1 across folds (Train vs Validation)")